In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

spark.sql("""
CREATE OR REPLACE TABLE gold.patient_encounter_summary AS
SELECT
    p.resource_id   AS patient_id,
    p.given_name, p.family_name, p.gender, p.birth_date,
    e.resource_id   AS encounter_id,
    e.status        AS encounter_status,
    e.period_start, e.period_end
FROM silver.patient_clean p
JOIN silver.encounter_clean e ON p.resource_id = e.patient_id
""")

spark.sql("""
CREATE OR REPLACE TABLE gold.encounter_observations AS
SELECT
    e.resource_id AS encounter_id, e.patient_id,
    o.resource_id AS observation_id, o.code, o.code_display,
    o.value_numeric, o.value_unit, o.value_string
FROM silver.encounter_clean e
JOIN silver.observation_clean o ON e.resource_id = o.encounter_id
""")

spark.sql("""
CREATE OR REPLACE TABLE gold.patient_conditions AS
SELECT
    p.resource_id AS patient_id, p.given_name, p.family_name,
    c.condition_code, c.condition_display, c.clinical_status, c.onset_date
FROM silver.patient_clean p
JOIN silver.condition_clean c ON p.resource_id = c.patient_id
""")

for t in ["patient_encounter_summary", "encounter_observations", "patient_conditions"]:
    print(t, spark.table(f"gold.{t}").count())

In [0]:
spark.sql('SELECT * FROM gold.patient_encounter_summary LIMIT 5').show()